# 튜닝: LightGBM 순위 회귀 — 규제·입력 구성·입력 변환

모델은 하나로 고정하고(LightGBM 순위 회귀, 정답 매달 백분위, MSE, 깊이 3, 학습률 0.01) **조건 하나씩만** 바꿔 192개월 평균 IC를 비교한다.
조정값(H3 1·3·5년, 트리 수 1~1000)은 지금처럼 해마다 직전 1년으로 고른다. 기준과 차이가 0.002 미만이면 단순한 쪽(기준)을 유지한다.

| 절 | 바꾸는 것 | 왜 보나 |
| --- | --- | --- |
| 1 기준 | 없음: 입력 [Size, B/M, Mom, σ], 순위 입력, L2 0, 잎 최소 행 1 | 아래 모든 비교의 기준 |
| 2 규제 | L2 1·5·10 / 잎 최소 행 20·100 | 깊이 3에서 학습−검증 IC 차이가 0.05로 큼. XGBoost 기본 L2 1, LightGBM 기본 잎 20, 100은 관례값 |
| 3 입력 구성 | [σ] → [B/M, σ] → [B/M, Mom, σ] → (기준) | 입력을 하나씩 더할 때 IC가 얼마나 오르는지(가설 기여) |
| 4 입력 변환 | z점수, 1·99% 윈저 후 z점수 | 선형은 z점수가 나았음. 트리도 그런지 |
| 5 학습 창 | 이동창 5·10년 / 2013년까지 한 번 학습한 모델로 2014~2021 예측 (선형 1등 설정도 같이) | 교수님 제안(이동창), 테스트는 재학습 없이 1~4년 앞을 예측하므로 모델이 낡는 정도 확인 |

- 실행 약 1.6시간(부스팅 학습 묶음 14번 × 약 6분 + 선형 4번 + 입력 준비 4번). 절마다 끝나면 표를 저장하므로 중간에 멈춰도 그때까지 결과는 남는다.
- 결과: `outputs/model_results/<AUTHOR>/tuning_lgb_rankreg.csv` (실험마다 한 줄).

In [ ]:
import os, sys, time
from pathlib import Path

AUTHOR = ''        # 결과 csv 위치 outputs/model_results/<AUTHOR>/ (영문)
N_THREADS = 6      # LightGBM CPU 스레드
FOLDS = None       # None = 16개 Fold 전부(시험할 때만 줄임)

if not AUTHOR:
    raise ValueError('AUTHOR를 채운다.')
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir() and (p / 'config' / 'experiment.yaml').is_file())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import yaml
from src.evaluation import model_test as mt

config = yaml.safe_load((ROOT / 'config' / 'experiment.yaml').read_text(encoding='utf-8'))
OUT_DIR = ROOT / 'outputs' / 'model_results' / AUTHOR
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / 'tuning_lgb_rankreg.csv'
SETTING = [('lgb_rankreg', 'percentile', 'mse', 3, 0.01)]
BASE_INPUTS = ['size', 'bm', 'mom', 'sigma']
ROWS = []
pd.set_option('display.width', 200, 'display.float_format', '{:.4f}'.format)


MONTHLY = {}       # 조건 → 목표월별 IC(5절에서 같은 달끼리 비교)


def monthly(res):
    """목표월별 IC. 한 달 예측이 모두 같으면 그 달 IC 0(우리 결과와 같은 계산)."""
    vk = res['val_keys']
    pred = next(iter(res['predictions'].values()))
    t = mt.monthly_ic(vk.target_month.to_numpy(), pred, vk.y.to_numpy(float))
    return t.ic.where(t.reason.ne('constant_score'), 0.0)


def score(ic):
    """평균, ICIR, IC>0 비율."""
    return {'months': int(ic.notna().sum()), 'mean_ic': ic.mean(), 'icir': ic.mean() / ic.std(), 'ic_pos': (ic > 0).mean()}


def run(prep, section, condition, *, inputs=BASE_INPUTS, overrides=None, transform='rank', folds=None):
    """한 조건으로 LightGBM 순위 회귀 깊이 3을 돌리고(기본 16 Fold) 한 줄을 ROWS에 더한 뒤 csv를 저장한다."""
    t0 = time.perf_counter()
    if folds is None:
        folds = mt.plan_folds(config['MODEL_GRID']) if FOLDS is None else FOLDS
    res = mt.run_boosting(prep, config, 'lgb_rankreg', phase='individual', settings=SETTING, sets=dict([mt.make_input_set(inputs)]),
                          folds=folds, overrides=overrides, n_threads=N_THREADS, log=lambda *a, **k: None)
    sel = res['selections']
    ic = monthly(res)
    MONTHLY[condition] = ic
    row = {'section': section, 'condition': condition, 'inputs': '+'.join(inputs), 'input_transform': transform,
           'overrides': str(overrides or {}), **score(ic), 'train_ic': sel.train_ic.mean(),
           'train_minus_valid': np.nan, 'trees_median': sel.trees.median(), 'trees_min': sel.trees.min(), 'trees_max': sel.trees.max(),
           'minutes': round((time.perf_counter() - t0) / 60, 1)}
    row['train_minus_valid'] = row['train_ic'] - row['mean_ic']
    ROWS.append(row)
    pd.DataFrame(ROWS).to_csv(OUT_CSV, index=False, encoding='utf-8-sig')
    print(f"[{section}] {condition}: 평균 IC {row['mean_ic']:.4f}, ICIR {row['icir']:.3f}, 학습−검증 {row['train_minus_valid']:.3f}, "
          f"트리 중앙 {row['trees_median']:.0f}, {row['minutes']}분")
    return row


print('ROOT:', ROOT, '| 결과:', OUT_CSV)

## 1. 기준 (약 8분)

순위 입력을 준비하고 기준 조건을 한 번 돌린다. 2·3절은 이 입력을 그대로 쓴다.

In [ ]:
t0 = time.perf_counter()
prep_rank = mt.prepare_inputs(ROOT, config)
print(f'입력 준비(순위) {time.perf_counter() - t0:.0f}초, 학습 시작 {prep_rank["checks"]["train_start_target_year"]}')
base = run(prep_rank, '1 기준', '기준')

## 2. 규제 (약 30분)

L2는 잎 값을 0 쪽으로 눌러 과적합을 줄이고, 잎 최소 행은 너무 적은 종목으로 잎을 만들지 못하게 한다.

In [ ]:
for cond, ov in [('L2 1', {'lambda_l2': 1.0}), ('L2 5', {'lambda_l2': 5.0}), ('L2 10', {'lambda_l2': 10.0}),
                 ('잎 최소 행 20', {'min_data_in_leaf': 20}), ('잎 최소 행 100', {'min_data_in_leaf': 100})]:
    run(prep_rank, '2 규제', cond, overrides=ov)

## 3. 입력 구성 (약 18분)

σ 하나에서 시작해 B/M, Mom을 차례로 더한다. 마지막 단계(+Size)는 1절 기준과 같다.

In [ ]:
for inputs in (['sigma'], ['bm', 'sigma'], ['bm', 'mom', 'sigma']):
    run(prep_rank, '3 입력 구성', '+'.join(inputs), inputs=inputs)

## 4. 입력 변환 (약 16분)

입력을 매달 z점수(평균 0, 표준편차 1)나 1·99%에서 자른 뒤 z점수로 바꾼다. 트리는 값의 순서만 쓰므로 차이가 작을 것으로 예상.

In [ ]:
for how in ('zscore', 'winsor_zscore'):
    t0 = time.perf_counter()
    prep = mt.prepare_inputs(ROOT, config, input_transform=how)
    print(f'입력 준비({how}) {time.perf_counter() - t0:.0f}초')
    run(prep, '4 입력 변환', how, transform=how)

## 5. 학습 창 (약 25분)

지금 방식(확장창: 2001년부터 해마다 1년씩 늘려 다시 학습)과 두 가지를 비교한다. 같은 조건을 선형 1등 설정(Ridge × 월별 z점수 × Huber①, z점수 입력)에도 돌려 선형·부스팅을 함께 본다(선형은 각자의 확장창 기준과 비교). 내부 검증 1년, 고른 뒤 재학습 없음은 그대로.

- **이동창 W년**: 내부 학습을 직전 W년으로 자른다(예: W=5, 2015년 예측 = 2008~2012 학습, 2013 내부 검증). W년보다 짧은 앞쪽 Fold는 확장창과 같다(W=5는 2008년, W=10은 2013년 예측부터 달라짐). 5·10은 문헌 값이 아닌 관례값.
- **고정 모델(~2013)**: 2001~2012 학습, 2013 내부 검증으로 한 번 만든 모델로 2014~2021을 재학습 없이 예측(= 지금 Fold 9 모델을 8년 내내 사용). 비교는 기준의 같은 96개월과, 연도별 차이(뒤로 갈수록 벌어지면 모델이 낡는다는 뜻)로 본다.


In [ ]:
PLAN = mt.plan_folds(config['MODEL_GRID'])
if FOLDS is not None:
    PLAN = [f for f in PLAN if f['fold'] in {g['fold'] for g in FOLDS}]


def window_folds(w):
    """이동창: 내부 학습 = 내부 검증 직전 w년(2001년보다 앞이면 2001년부터)."""
    out = []
    for f in PLAN:
        lo = max(f['train_years'][0], f['valid_year'] - 1 - w)
        out.append({**f, 'train_years': (lo, f['valid_year'] - 2), 'inner_train_years': f['valid_year'] - 1 - lo})
    return out


FIXED_LAST_TRAIN = 2012   # 내부 학습 ~2012, 내부 검증 2013 → 2014~2021 예측


def fixed_folds():
    """고정 모델: 바깥 검증 2014~2021의 모든 Fold가 같은 학습·내부 검증을 쓴다(같은 모델을 Fold마다 다시 만듦, 난수 없어 동일)."""
    start = PLAN[0]['train_years'][0]
    return [{**f, 'train_years': (start, FIXED_LAST_TRAIN), 'inner_valid_year': FIXED_LAST_TRAIN + 1,
             'inner_train_years': FIXED_LAST_TRAIN - start + 1} for f in PLAN if f['valid_year'] > FIXED_LAST_TRAIN + 1]


WINDOWS = {'이동창 5년': window_folds(5), '이동창 10년': window_folds(10), '고정 모델(~2013)': fixed_folds()}
for cond, folds in WINDOWS.items():
    run(prep_rank, '5 학습 창', cond, folds=folds)

# 선형 1등 설정: 확장창 기준 + 같은 세 조건
LIN_SETTING = [('ridge', 'monthly_zscore', 'huber_q999')]
t0 = time.perf_counter()
prep_z = mt.prepare_inputs(ROOT, config, input_transform='zscore')
print(f'입력 준비(z점수) {time.perf_counter() - t0:.0f}초')


def run_lin(condition, folds):
    t0 = time.perf_counter()
    res = mt.run_linear(prep_z, config, folds=folds, settings=LIN_SETTING, sets=dict([mt.make_input_set(BASE_INPUTS)]),
                        log=lambda *a, **k: None)
    sel, ic = res['selections'], monthly(res)
    MONTHLY[condition] = ic
    row = {'section': '5 학습 창 (선형)', 'condition': condition, 'inputs': '+'.join(BASE_INPUTS), 'input_transform': 'zscore',
           'overrides': 'ridge·monthly_zscore·huber_q999', **score(ic), 'train_ic': sel.train_ic.mean(),
           'minutes': round((time.perf_counter() - t0) / 60, 1)}
    row['train_minus_valid'] = row['train_ic'] - row['mean_ic']
    ROWS.append(row)
    pd.DataFrame(ROWS).to_csv(OUT_CSV, index=False, encoding='utf-8-sig')
    print(f"[선형] {condition}: 평균 IC {row['mean_ic']:.4f}, ICIR {row['icir']:.3f}, 학습−검증 {row['train_minus_valid']:.3f}, {row['minutes']}분")


run_lin('선형 기준', PLAN)
for cond, folds in WINDOWS.items():
    run_lin('선형 ' + cond, folds)

year_of = lambda ic: pd.Series(ic.to_numpy(), index=pd.to_datetime(ic.index.astype(str)).year)
base_y = year_of(MONTHLY['기준'])
by_year = pd.DataFrame({'기준': base_y.groupby(level=0).mean()})
by_year['선형 기준'] = year_of(MONTHLY['선형 기준']).groupby(level=0).mean()
for cond in WINDOWS:
    by_year[cond + ' − 기준'] = year_of(MONTHLY[cond]).groupby(level=0).mean() - by_year['기준']
    by_year['선형 ' + cond + ' − 선형 기준'] = year_of(MONTHLY['선형 ' + cond]).groupby(level=0).mean() - by_year['선형 기준']
for pre, base_name in (('', '기준'), ('선형 ', '선형 기준')):
    fixed = MONTHLY[pre + '고정 모델(~2013)']
    print(f"{pre or 'LightGBM '}고정 모델 2014~2021: {fixed.mean():.4f} vs {base_name} 같은 달 {MONTHLY[base_name].reindex(fixed.index).mean():.4f}")
by_year.to_csv(OUT_DIR / 'tuning_lgb_rankreg_window_by_year.csv', encoding='utf-8-sig')
by_year


## 6. 비교

기준과의 차이(고정 모델은 기준의 같은 2014~2021 달과 비교). 0.002를 넘으면 그 조건을 검토한다.

In [ ]:
res = pd.DataFrame(ROWS)
base_of = lambda cond: '선형 기준' if cond.startswith('선형') else '기준'
base_on = lambda cond: MONTHLY[base_of(cond)].reindex(MONTHLY[cond].index).mean()
res['diff_vs_base'] = [r.mean_ic - base_on(r.condition) for r in res.itertuples()]
res['판단'] = np.where(res.condition.isin(['기준', '선형 기준']), '기준', np.where(res.diff_vs_base > 0.002, '검토', '기준 유지'))
res[['section', 'condition', 'mean_ic', 'diff_vs_base', 'icir', 'ic_pos', 'train_ic', 'train_minus_valid', 'trees_median', 'minutes', '판단']]